# Single-Clone CNV Analysis - Fusion vs Control vs Parental

- Use environment_CNandRNAnotebooks.yml
- Edit the 1st cell below to run for either HCC1806 or MDA-MB-231

## Samples: Control clones (C1-C8), Fusion clones (F1-F8), Parental lines (GFP, MC)

This notebook processes FREEC copy-number output for:
- **8 Control clones** (C1-C8): single-color (GFP-only or mCherry-only) clones isolated and expanded from single cells, treated as independent observations of the unfused state
- **8 Fusion clones** (F1-F8): double-positive (GFP+mCherry) clones isolated and expanded from single fusion events, treated as independent observations of the fused/WGD state
- **2 Parental bulk populations** (GFP, MC): the heterogeneous, unsorted starting populations, used as a pre-sorting reference

Genomic bins are filtered with the ENCODE hg38 blacklist (Boyle Lab) to retain only reliably mappable regions. No cross-sample variance filter or smoothing step is used - the blacklist replaces both, following the same rationale used for the bulk-population analysis (Scheinin et al. 2014 / QDNAseq).

**Statistical note - clones as pseudo-replicates:** the t-tests below treat each clone as an independent replicate. This is standard practice for clonal CNV studies and tests whether the *distribution* of copy-number values across clones differs between groups. Interpret results as clone-level variation, not classical biological replication.

**Parental group (n=2):** GFP and MC together give n=2 for any comparison involving Parental. This is the statistical minimum, and results involving Parental should be treated as suggestive rather than definitive. This is flagged again wherever it applies below.

**Analysis structure:**
- Part 1: Log2 copy number ratio (log2 CNR) - loading, blacklist filtering, clustering, genome plots
- Part 2: Integer copy number (CN) - loading, blacklist filtering, clustering, genome plots, ploidy distribution
- Part 3: Intra-group CN heterogeneity score
- Part 4: Chromosomal instability (CIN) metrics - FGA, TAI, mTAI, breakpoint count, CNA score, altered base pairs
- Part 5: Chromosome-arm level analysis - aggregation, delta heatmaps, perfect-doubling comparison, arm-level t-tests

**Configuration notes:**
- Baseline ploidy assumptions live in `GROUP_PLOIDY` in the configuration cell below (default: Control=2, Fusion=4, Parental=2). To reuse this notebook for a different cell line, update `GROUP_PLOIDY` (and `SAMPLE_PLOIDY_OVERRIDE` for any sample-specific exceptions), plus `RATIO_DIR`.
- `CIN_INCLUDE_PARENTAL_IN_STATS` controls whether Parental is included in the CIN metric boxplots and pairwise statistical comparisons in Part 4. The full CIN metrics table always includes Parental regardless of this toggle.
- Group colors (`GROUP_COLORS`) and figure style (font, size, output format) are both set in the configuration cell.


## 1  Configuration


In [ ]:
#(Make sure to comment out the set of lines below for the cell line not being analyzed)

## ---------- Use the 2 lines below for HCC1806 --------------
RATIO_DIR = "/stor/work/Brock/kennedy/SC_repo/data/WGS_CNV_Analysis/FREEC_Output_1806"
GROUP_PLOIDY = {
    "C": 2,   # Control clones - unfused, single-color
    "F": 4,   # Fusion clones - GFP+mCherry double-positive
    "P": 2,   # Parental bulk populations - unsorted, unfused
}

## ---------- Use the 2 lines below for MDA-MB-231 -----------
# RATIO_DIR = "/stor/work/Brock/kennedy/SC_repo/data/WGS_CNV_Analysis/FREEC_Output_231"
# GROUP_PLOIDY = {
#     "C": 3,   # Control clones - unfused, single-color
#     "F": 6,   # Fusion clones - GFP+mCherry double-positive
#     "P": 3,   # Parental bulk populations - unsorted, unfused
# }

In [ ]:
import os, glob, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib as mpl
from matplotlib.patches import Patch
import seaborn as sns
from collections import defaultdict
from itertools import combinations
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import pathlib
from matplotlib import font_manager, rcParams
warnings.filterwarnings("ignore")

# -- Chromosomes to retain ----------------------------------------------------
ALLOWED_CHROMS = [str(i) for i in range(1, 23)] + ["X"]

# -- FREEC window size ---------------------------------------------------------
WINDOW_SIZE = 500000   # 500 kb - must match window= in FREEC config

# ==============================================================================
#  PLOIDY CONTROLS - edit here to reuse this notebook for a different cell line
# ==============================================================================

#  SAMPLE_PLOIDY_OVERRIDE - optional per-sample override (short name -> ploidy)
#  Leave empty ({}) to use GROUP_PLOIDY for everything.
#  Example: one fusion clone that only partially doubled ->
#    "F5": 3
SAMPLE_PLOIDY_OVERRIDE = {
    # "F5": 3,
}
# ==============================================================================

# ==============================================================================
#  COLOUR CONTROLS - edit here to change plot colours
# ==============================================================================
#
#  GROUP_COLORS - one colour per group; applies to all samples in that group
#    "C"  ->  Control clones   (C1-C8)
#    "F"  ->  Fusion clones    (F1-F8)
#    "P"  ->  Parental lines   (GFP, MC)
#
GROUP_COLORS = {
    "C": "black",
    "F": "saddlebrown",
    "P": "royalblue",
}

#  SAMPLE_COLORS - optional per-sample overrides (short name -> colour)
#  Leave empty ({}) to use GROUP_COLORS for everything.
SAMPLE_COLORS = {
    # "F3": "red",
}
# ==============================================================================

GROUP_NAMES = {
    "C": "Control clones",
    "F": "Fusion clones",
    "P": "Parental",
}

# -- Display order --------------------------------------------------------------
SAMPLE_GROUP_ORDER = ["C", "F", "P"]

# -- ENCODE hg38 blacklist --------------------------------------------------------
# Removes pericentromeric, telomeric, and other low-mappability artifact-prone
# regions. Replaces any cross-sample variance filter or smoothing step, following
# the same approach used for the bulk-population analysis (Scheinin et al. 2014
# / QDNAseq). The local path is used as a cache so the file is only downloaded once.
BLACKLIST_URL = "https://github.com/Boyle-Lab/Blacklist/raw/master/lists/hg38-blacklist.v2.bed.gz"

# -- CIN metric parameters ---------------------------------------------------------
CNA_THRESHOLD = 0.2

# -- Statistical threshold ------------------------------------------------------------
ALPHA = 0.05

# -- Reference genome -------------------------------------------------------------------
GENOME   = "hg38"

# -- CIN metrics Parental toggle (Part 4) -----------------------------------------------
# The saved CIN metrics table always includes Parental. This toggle only controls
# whether Parental is included in the boxplots and pairwise significance tests.
# Set to False to drop Parental from those comparisons (n=2 gives low power there).
CIN_INCLUDE_PARENTAL_IN_STATS = False

# -- Plot toggles -----------------------------------------------------------------------
SHOW_GRID          = False
SHOW_STAT_BRACKETS = True
BOXPLOT_FILL       = "color"   # "color" or "outline"
BOX_ALPHA          = 0.45

# -- Output directory ---------------------------------------------------------------------
OUTPUT_DIR = "cnv_analysis_output_1806_SC"
os.makedirs(OUTPUT_DIR, exist_ok=True)
BLACKLIST_LOCAL = os.path.join(OUTPUT_DIR, "hg38-blacklist.v2.bed.gz")
print("Output directory:", os.path.abspath(OUTPUT_DIR))

# ==============================================================================
#  FIGURE STYLE SETTINGS
# ==============================================================================
# FONT_DIR = pathlib.Path('/stor/work/Brock/kennedy/fonts/arial')
# if FONT_DIR.exists():
#     for _fp in list(FONT_DIR.glob('*.TTF')) + list(FONT_DIR.glob('*.ttf')):
#         font_manager.fontManager.addfont(str(_fp))
#     _avail = [f.name for f in font_manager.fontManager.ttflist]
#     if 'Arial' in _avail:
#         print(f"Arial loaded from {FONT_DIR}")
#     else:
#         print(f"Font files found in {FONT_DIR} but Arial was not registered - using default font.")
# else:
#     print(f"Font dir not found ({FONT_DIR}) - using default font.")

# FONT_FAMILY = "Arial"
FONT_SIZE   = 16
FONT_BOLD   = False
FIG_EXT     = ".png"   # ".svg" or ".png"
FIG_DPI     = 300       # only used when FIG_EXT == ".png"

def apply_style():
    w = "bold" if FONT_BOLD else "normal"
    rcParams.update({
        # "font.family":           FONT_FAMILY,
        "font.size":             FONT_SIZE,
        "font.weight":           w,
        "axes.titlesize":        FONT_SIZE + 1,
        "axes.titleweight":      w,
        "axes.labelsize":        FONT_SIZE,
        "axes.labelweight":      w,
        "xtick.labelsize":       FONT_SIZE - 1,
        "ytick.labelsize":       FONT_SIZE - 1,
        "legend.fontsize":       FONT_SIZE - 1,
        "legend.title_fontsize": FONT_SIZE,
        "figure.titlesize":      FONT_SIZE + 2,
        "figure.titleweight":    w,
    })

apply_style()

def save_fig(fig, name, tight=True):
    path = os.path.join(OUTPUT_DIR, f"{name}{FIG_EXT}")
    kw = {"bbox_inches": "tight"} if tight else {}
    if FIG_EXT.lower() == ".png":
        kw["dpi"] = FIG_DPI
    fig.savefig(path, **kw)
    print(f"Saved: {path}")
    return path

print(f"Style applied  |  FIG_EXT={FIG_EXT}  |  FONT_SIZE={FONT_SIZE}  |  FONT_BOLD={FONT_BOLD}")
# ==============================================================================
#  ADDITIONAL ANALYSIS CONTROLS (permutation tests, segment reconstruction)
# ==============================================================================

# -- Permutation test settings (Part 3.3) ---------------------------------------
N_PERMUTATIONS = 10000
RANDOM_SEED    = 0

print(f"Additional analysis controls set  |  N_PERMUTATIONS={N_PERMUTATIONS}  |  RANDOM_SEED={RANDOM_SEED}")


## 2  Helper Functions
*Run this cell before any analysis cell.*


In [ ]:
# ------------------------------------------------------------------------------
# Sample-name parsing (adapted for naming scheme)
# ------------------------------------------------------------------------------

def get_display_name(sample_name):
    """
    Return the short label used in all plot titles and axis labels.
    'C1_1806_PT_p...' -> 'C1'
    'GFP_1806_PT_...' -> 'GFP'
    'MC_1806_PT_...'  -> 'MC'
    """
    return sample_name.split("_")[0]

def get_group(sample_name):
    """Return the group key for a sample name."""
    short = get_display_name(sample_name)
    if re.match(r'^C\d', short): return "C"
    if re.match(r'^F\d', short): return "F"
    if short in ("GFP", "MC"):   return "P"
    return short  # fallback

def get_clone_number(sample_name):
    """Extract the numeric clone ID from C1/F8 etc., or 0 if not applicable."""
    m = re.match(r'^[CF](\d+)', get_display_name(sample_name))
    return int(m.group(1)) if m else 0

def get_color(sample_name):
    """Return colour for a sample, respecting per-sample SAMPLE_COLORS overrides."""
    short = get_display_name(sample_name)
    if short in SAMPLE_COLORS:
        return SAMPLE_COLORS[short]
    return GROUP_COLORS.get(get_group(sample_name), "gray")

def sort_sample_cols(sample_cols):
    """Sort columns in SAMPLE_GROUP_ORDER, then numerically by clone number."""
    def sort_key(s):
        grp     = get_group(s)
        grp_idx = SAMPLE_GROUP_ORDER.index(grp) if grp in SAMPLE_GROUP_ORDER else 99
        return (grp_idx, get_clone_number(s), s)
    return sorted(sample_cols, key=sort_key)

def get_ploidy(sample_name):
    """
    Return the baseline ploidy assumed for a sample, used by the CIN metrics
    in Part 4. Checks SAMPLE_PLOIDY_OVERRIDE first, then falls back to
    GROUP_PLOIDY by group.
    """
    short = get_display_name(sample_name)
    if short in SAMPLE_PLOIDY_OVERRIDE:
        return SAMPLE_PLOIDY_OVERRIDE[short]
    return GROUP_PLOIDY.get(get_group(sample_name), 2)

# ------------------------------------------------------------------------------
# Genomic coordinate helpers
# ------------------------------------------------------------------------------

def chrom_sort_key(chrom):
    c = str(chrom).replace("chr", "")
    if c.isdigit(): return int(c)
    return 23 if c.upper() == "X" else 24

def sort_df_by_genome(df):
    df = df.copy()
    df["_cs"] = df["Chromosome"].apply(chrom_sort_key)
    return df.sort_values(["_cs", "Start"]).drop(columns="_cs").reset_index(drop=True)

def get_chrom_layout(bin_coords_df):
    df = bin_coords_df.copy().reset_index(drop=True)
    df["Plot_Index"] = range(len(df))
    chrom_breaks, label_positions, chrom_labels = [], [], []
    first = True
    for chrom, grp in df.groupby("Chromosome", sort=False):
        idx = grp["Plot_Index"].values
        if not len(idx): continue
        chrom_labels.append(str(chrom).replace("chr", ""))
        label_positions.append(int((idx[0] + idx[-1]) // 2))
        if not first: chrom_breaks.append(int(idx[0]))
        first = False
    return df, chrom_breaks, label_positions, chrom_labels

def build_legend_patches(sample_cols):
    groups = {get_group(s) for s in sample_cols}
    return [mpatches.Patch(color=GROUP_COLORS.get(g, "gray"),
                           label=GROUP_NAMES.get(g, g))
            for g in SAMPLE_GROUP_ORDER if g in groups]

# ------------------------------------------------------------------------------
# ENCODE blacklist loading and filtering
# ------------------------------------------------------------------------------

def load_blacklist(url=None, local_path=None):
    """
    Load the ENCODE hg38 blacklist BED file.
    Tries local cache first; falls back to downloading from url.
    Returns a DataFrame with columns: chrom_clean, start, end.
    """
    import urllib.request
    path = local_path or BLACKLIST_LOCAL
    if not os.path.exists(path):
        src = url or BLACKLIST_URL
        print(f"Downloading blacklist from {src} ...")
        urllib.request.urlretrieve(src, path)
        print(f"Saved to {path}")
    df = pd.read_csv(path, sep="\t", header=None, compression="gzip",
                     usecols=[0,1,2], names=["chrom","start","end"])
    df["chrom_clean"] = df["chrom"].str.replace("chr", "", regex=False)
    print(f"Blacklist loaded: {len(df)} regions")
    return df

def filter_blacklist(df, blacklist_df, window_size=WINDOW_SIZE):
    """
    Remove bins from df that overlap any region in blacklist_df.
    df must have columns Chromosome and Start (0-based).
    Overlap condition: bl_start < bin_end AND bl_end > bin_start.
    Returns a filtered DataFrame with reset index.
    """
    result_parts = []
    for chrom, bin_grp in df.groupby("Chromosome"):
        c = str(chrom).replace("chr", "")
        bl = blacklist_df[blacklist_df["chrom_clean"] == c]
        if bl.empty:
            result_parts.append(bin_grp)
            continue
        starts    = bin_grp["Start"].values.astype(int)
        ends      = starts + window_size
        keep      = np.ones(len(starts), dtype=bool)
        bl_starts = bl["start"].values.astype(int)
        bl_ends   = bl["end"].values.astype(int)
        for bs, be in zip(bl_starts, bl_ends):
            keep &= ~((bs < ends) & (be > starts))
        result_parts.append(bin_grp[keep])
    filtered = pd.concat(result_parts).reset_index(drop=True)
    return filtered

# ------------------------------------------------------------------------------
# Centromere loading (for arm-level analysis, Part 5)
# ------------------------------------------------------------------------------

_CENTROMERE_URLS = {
    "hg38": "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/centromeres.txt.gz",
    "hg19": "https://hgdownload.soe.ucsc.edu/goldenPath/hg19/database/gap.txt.gz",
}
_HG38_FALLBACK = {
    "1":121500000,"2":92000000,"3":90900000,"4":49700000,"5":46500000,
    "6":58800000,"7":60100000,"8":45200000,"9":43000000,"10":39800000,
    "11":53400000,"12":35500000,"13":17700000,"14":17200000,"15":19000000,
    "16":36800000,"17":25100000,"18":18500000,"19":26200000,"20":28100000,
    "21":12000000,"22":15000000,"X":61000000,
}

def load_centromeres(genome="hg38"):
    try:
        url = _CENTROMERE_URLS[genome]
        df  = pd.read_csv(url, sep="\t", header=None, compression="gzip")
        if genome == "hg38":
            df.columns = ["bin","chrom","start","end","name"]
        else:
            df.columns = ["bin","chrom","start","end","ix","n","size","type","bridge"]
            df = df[df["type"] == "centromere"]
        df["chrom_clean"] = df["chrom"].str.replace("chr", "", regex=False)
        centro = {}
        for chrom, grp in df.groupby("chrom_clean"):
            centro[chrom] = int((grp["start"].min() + grp["end"].max()) / 2)
        print(f"Centromere positions loaded from UCSC ({genome}): {len(centro)} chromosomes")
        return centro
    except Exception as e:
        print(f"UCSC fetch failed ({e}); using hg38 hardcoded fallback.")
        return _HG38_FALLBACK

print("Helper functions loaded.")
print("Display name examples:")
for ex in ["C1_1806_PT_p_example", "F8_1806_PT_p_example", "GFP_1806_PT_example", "MC_1806_PT_example"]:
    print(f"  {ex!r:40s} -> {get_display_name(ex)!r}  group={get_group(ex)}  ploidy={get_ploidy(ex)}")


---
# Part 1 - Log2 Copy Number Ratio (log2 CNR)
*Source column:* `Ratio` from `_ratio.txt` (FREEC Stage 1).
*Used for:* hierarchical clustering, individual genome plots, per-chromosome plots, and arm-level analysis.
*Processing:* unmappable bin removal -> ENCODE blacklist filtering -> log2 transform. No smoothing or cross-sample variance filter is applied.
The resulting `log2_filtered_ratio_matrix.tsv` is used for all log2 CNR analyses in this notebook.


### 1.1  Merge FREEC Ratio Data


In [ ]:
ratio_files = glob.glob(os.path.join(RATIO_DIR, "**/*_ratio.txt"), recursive=True)
if not ratio_files:
    ratio_files = glob.glob(os.path.join(RATIO_DIR, "*_ratio.txt"))

print(f"Found {len(ratio_files)} ratio file(s):")
dfs_ratio = []
for f in sorted(ratio_files):
    sample_name = os.path.basename(f).split("_paired")[0]
    df = pd.read_csv(f, sep="\t",
                     usecols=["Chromosome", "Start", "Ratio"],
                     dtype={"Chromosome": str})
    df = df[df["Chromosome"].isin(ALLOWED_CHROMS)]
    df = df.rename(columns={"Ratio": sample_name})
    dfs_ratio.append(df)
    grp = get_group(sample_name)
    print(f"  {get_display_name(sample_name):<8}  ({sample_name})  group={GROUP_NAMES.get(grp, grp)}")

merged_ratio = dfs_ratio[0]
for df in dfs_ratio[1:]:
    merged_ratio = pd.merge(merged_ratio, df, on=["Chromosome", "Start"], how="inner")
merged_ratio = sort_df_by_genome(merged_ratio)

out = os.path.join(OUTPUT_DIR, "merged_ratio_matrix.tsv")
merged_ratio.to_csv(out, sep="\t", index=False)
print(f"\nSaved: {out}  [{merged_ratio.shape[0]} bins x {len(dfs_ratio)} samples]")
sample_cols_ratio = [c for c in merged_ratio.columns if c not in ["Chromosome", "Start"]]


### 1.2  Remove Unmappable Bins, Apply ENCODE Blacklist, and Log2-Transform


In [ ]:
df_r  = pd.read_csv(os.path.join(OUTPUT_DIR, "merged_ratio_matrix.tsv"),
                    sep="\t", dtype={"Chromosome": str})
scols = [c for c in df_r.columns if c not in ["Chromosome", "Start"]]

# Step 1: remove unmappable bins (-1 sentinel)
mask_bad   = (df_r[scols] == -1).any(axis=1)
df_r_clean = df_r[~mask_bad].copy()
print(f"Bins before -1 filter        : {len(df_r)}")
print(f"Bins after  -1 filter        : {len(df_r_clean)}  (removed {mask_bad.sum()})")

# Step 2: remove ENCODE blacklisted bins.
# Replaces any cross-sample variance filter or smoothing step. Removes
# pericentromeric, telomeric, and other low-mappability artifact-prone regions
# without biasing against biologically variable bins the way a variance filter
# would (Scheinin et al. 2014 / QDNAseq).
BLACKLIST  = load_blacklist()
n_before   = len(df_r_clean)
df_r_clean = filter_blacklist(df_r_clean, BLACKLIST, window_size=WINDOW_SIZE)
print(f"Bins after  blacklist filter : {len(df_r_clean)}  (removed {n_before - len(df_r_clean)})")

# Step 3: log2-transform - Ratio is observed/expected so log2(Ratio) = log2 CNR
df_r_clean[scols] = np.log2(df_r_clean[scols].replace(0, np.nan))
df_r_clean = sort_df_by_genome(df_r_clean)
out = os.path.join(OUTPUT_DIR, "log2_filtered_ratio_matrix.tsv")
df_r_clean.to_csv(out, sep="\t", index=False)
print(f"Saved: {out}")
print("This matrix (blacklisted, -1 filtered, log2 transformed, unsmoothed) is used")
print("for all log2 CNR analyses: clustering, t-tests, heterogeneity, and arm-level work.")


### 1.3  Hierarchical Clustering - Log2 CNR
Two linkage strategies are shown:
- **Euclidean / Ward** - clusters by absolute profile distance.
- **Correlation / Average** - clusters by profile shape, independent of overall offset.


#### 1.3a  Euclidean / Ward


In [ ]:
def plot_clustermap(matrix_file, metric, method, title_suffix, out_name):
    cnv     = pd.read_csv(matrix_file, sep="\t", dtype={"Chromosome": str})
    cnv_num = cnv.drop(columns=["Chromosome", "Start"]).apply(pd.to_numeric, errors="coerce")
    samps   = sort_sample_cols(cnv_num.columns.tolist())
    cnv_num = cnv_num[samps]

    # Drop bins that are NaN in any sample (e.g. log2(0) -> NaN from the log2
    # transform in cell 9); scipy's linkage() requires all-finite input.
    n_before = len(cnv_num)
    cnv_num  = cnv_num.dropna(axis=0, how="any")
    n_after  = len(cnv_num)
    if n_after < n_before:
        print(f"[{out_name}] Dropped {n_before - n_after} bin(s) with NaN values "
              f"before clustering ({n_after} bins remain).")
    cnv = cnv.loc[cnv_num.index]

    display_names  = [get_display_name(s) for s in samps]
    cnv_num_disp   = cnv_num.copy()
    cnv_num_disp.columns = display_names
    row_colors     = pd.Series([get_color(s) for s in samps], index=display_names)
    legend_patches = build_legend_patches(samps)

    g = sns.clustermap(
        cnv_num_disp.T,
        cmap="coolwarm", metric=metric, method=method,
        vmin=-1, vmax=1,
        row_cluster=True, col_cluster=False,
        row_colors=row_colors,
        figsize=(16, max(8, len(samps) * 0.55)),
        cbar_pos=None,
        dendrogram_ratio=(0.12, 0), colors_ratio=0.025,
    )
    g.fig.subplots_adjust(right=0.80)
    g.ax_col_dendrogram.set_visible(False)

    # Row-colour annotation strip (left of the heatmap) draws its own x-axis
    # by default, which shows up as a stray tick beneath the group colour
    # column. It carries no information of its own, so hide it entirely.
    g.ax_row_colors.set_xticks([])
    g.ax_row_colors.tick_params(bottom=False, labelbottom=False)

    bin_info = cnv[["Chromosome", "Start"]].reset_index(drop=True)
    bin_info["idx"] = range(len(bin_info))
    clpos, cltxt, cdivs, prev = [], [], [], None
    for chrom, grp in bin_info.groupby("Chromosome", sort=False):
        iv = grp["idx"].values
        clpos.append(float(iv[0] + iv[-1]) / 2)
        cltxt.append(str(chrom).replace("chr", ""))
        if prev is not None: cdivs.append(float(iv[0]))
        prev = chrom
    g.ax_heatmap.set_xticks(clpos)
    g.ax_heatmap.set_xticklabels(cltxt, rotation=90, fontsize=max(6, FONT_SIZE - 2))
    g.ax_heatmap.set_xlabel("Chromosome", fontsize=FONT_SIZE - 2)
    for xd in cdivs:
        g.ax_heatmap.axvline(xd, color="black", linestyle="--", alpha=0.25, linewidth=0.5)

    ordered_names = cnv_num_disp.T.index[g.dendrogram_row.reordered_ind]
    g.ax_heatmap.set_yticks(np.arange(len(ordered_names)) + 0.5)
    g.ax_heatmap.set_yticklabels(ordered_names, fontsize=FONT_SIZE - 2,
                                 rotation=0, va="center")

    for yd in range(1, len(ordered_names)):
        g.ax_heatmap.axhline(yd, color="black", linestyle="--", alpha=0.25, linewidth=0.5)

    cbar_ax = g.fig.add_axes([0.85, 0.62, 0.018, 0.22])
    norm    = mpl.colors.Normalize(vmin=-1, vmax=1)
    cb      = mpl.colorbar.ColorbarBase(cbar_ax, cmap=plt.get_cmap("coolwarm"),
                                        norm=norm, orientation="vertical")
    cb.set_label(r"$\log_2$CNR", fontsize=FONT_SIZE - 2)
    g.fig.legend(handles=legend_patches, title="Group",
                 fontsize=FONT_SIZE - 2, title_fontsize=FONT_SIZE - 2,
                 loc="upper right", bbox_to_anchor=(0.995, 0.52),
                 bbox_transform=g.fig.transFigure,
                 frameon=True, framealpha=0.95)
    g.fig.suptitle(
        f"Hierarchical Clustering - Log2 CNR ({method.title()} / {metric.title()})\n{title_suffix}",
        y=1.01, fontsize=FONT_SIZE - 2)
    save_fig(g.fig, out_name)
    plt.show()

plot_clustermap(
    os.path.join(OUTPUT_DIR, "log2_filtered_ratio_matrix.tsv"),
    metric="euclidean", method="ward",
    title_suffix="",
    out_name="clustering_log2cnr_ward_euclidean",
)

#### 1.3b  Correlation / Average


In [ ]:
plot_clustermap(
    os.path.join(OUTPUT_DIR, "log2_filtered_ratio_matrix.tsv"),
    metric="correlation", method="average",
    title_suffix="",
    out_name="clustering_log2cnr_avg_correlation",
)


### 1.4  Individual Sample Genome Plots - Log2 CNR


In [ ]:
def plot_individual_samples(matrix_file, value_label="Log2 CNR",
                            y_range=(-2, 2), output_prefix="log2cnr",
                            title_suffix="Log2 Copy Number Ratio"):
    df    = pd.read_csv(matrix_file, sep="\t", dtype={"Chromosome": str})
    df    = sort_df_by_genome(df)
    scols = sort_sample_cols([c for c in df.columns if c not in ["Chromosome", "Start"]])
    bin_df, chrom_breaks, label_pos, chrom_labels = get_chrom_layout(df[["Chromosome","Start"]])
    df    = df.merge(bin_df[["Chromosome","Start","Plot_Index"]], on=["Chromosome","Start"], how="left")

    n = len(scols); ncols = 3; nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.2 * nrows), sharex=True, sharey=True)
    axes = np.array(axes).flatten()

    for i, s in enumerate(scols):
        ax = axes[i]
        ax.scatter(df["Plot_Index"], df[s], color=get_color(s), s=4, alpha=0.65, linewidths=0)
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
        for x in chrom_breaks:
            ax.axvline(x, color="black", linestyle="--", alpha=0.25, linewidth=0.6)
        ax.set_xticks(label_pos)
        ax.set_xticklabels(chrom_labels, rotation=90, fontsize=max(6, FONT_SIZE - 10))
        ax.set_ylabel(value_label, fontsize=max(6, FONT_SIZE - 9))
        ax.set_xlabel("Chromosome", fontsize=max(6, FONT_SIZE - 9))
        ax.set_title(get_display_name(s), fontsize=max(6, FONT_SIZE - 6), color=get_color(s), fontweight="bold")
        if y_range: ax.set_ylim(y_range)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.legend(handles=build_legend_patches(scols), title="Group",
               fontsize=max(6, FONT_SIZE - 7), loc="lower right", bbox_to_anchor=(1.0, 0.0),
               frameon=True, framealpha=0.9)
    fig.suptitle(f"Sample CNV Profiles - {title_suffix}", fontsize=max(6, FONT_SIZE - 3), y=1.01)
    plt.tight_layout()
    save_fig(fig, f"{output_prefix}_sample_profiles")
    plt.show()

plot_individual_samples(
    os.path.join(OUTPUT_DIR, "log2_filtered_ratio_matrix.tsv"),
    value_label="Log2 CNR", y_range=(-2, 2),
    output_prefix="log2cnr", title_suffix="Log2 Copy Number Ratio",
)


### 1.5  Per-Chromosome Heatmap - Log2 CNR


In [ ]:
def plot_chromosome_heatmap(matrix_file, chromosome,
                            value_label="Log2 CNR", vmin=-1.0, vmax=1.0,
                            cmap="RdBu_r", output_prefix="chr_log2cnr"):
    df        = pd.read_csv(matrix_file, sep="\t", dtype={"Chromosome": str})
    chrom_str = str(chromosome).replace("chr", "")
    df_chr    = df[df["Chromosome"].apply(lambda c: str(c).replace("chr","")) == chrom_str].copy()
    if df_chr.empty:
        print(f"No data for chromosome {chrom_str}"); return

    df_chr = sort_df_by_genome(df_chr).reset_index(drop=True)
    scols  = sort_sample_cols([c for c in df_chr.columns if c not in ["Chromosome","Start"]])
    n      = len(scols)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(3, 0.55*n+1.2)), squeeze=False)
    fig.subplots_adjust(hspace=0.05, right=0.86)

    for i, s in enumerate(scols):
        ax  = axes[i, 0]
        img = np.array([df_chr[s].to_numpy(dtype=float)])
        ax.imshow(img, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap, interpolation="nearest")
        ax.set_yticks([]); ax.set_xticks([])
        ax.set_ylabel(get_display_name(s), fontsize=max(6, FONT_SIZE), rotation=0,
                      ha="right", va="center", labelpad=4,
                      color=get_color(s), fontweight="bold")
        for sp in ax.spines.values():
            sp.set_linewidth(0.6); sp.set_edgecolor(get_color(s))

    cbar_ax = fig.add_axes([0.88, 0.12, 0.025, 0.76])
    fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax)),
                 cax=cbar_ax, label=value_label)
    fig.suptitle(f"Chromosome {chrom_str} - {value_label}", fontsize=max(6, FONT_SIZE - 4), fontweight="bold", y=1.01)
    save_fig(fig, f"{output_prefix}_chr{chrom_str}")
    plt.show()

for chrom in [str(i) for i in range(1, 23)] + ["X"]:
    plot_chromosome_heatmap(
        os.path.join(OUTPUT_DIR, "log2_filtered_ratio_matrix.tsv"),
        chromosome=chrom, value_label=r"log$_2$ CNR", vmin=-1, vmax=1,
        output_prefix="chr_log2cnr",
    )


---
# Part 2 - Integer Copy Number (CN)
*Source column:* `CopyNumber` from `_ratio.txt` (FREEC Stage 2).
*Used for:* CN-level clustering, individual genome plots, ploidy distribution, heterogeneity score, and arm-level analysis.
*Note:* CN is not log2-transformed - integer states are preserved for per-bin biological interpretation.


### 2.1  Merge FREEC Copy Number Data


In [ ]:
dfs_cn = []
for f in sorted(ratio_files):
    sample_name = os.path.basename(f).split("_paired")[0]
    df = pd.read_csv(f, sep="\t",
                     usecols=["Chromosome", "Start", "CopyNumber"],
                     dtype={"Chromosome": str})
    df = df[df["Chromosome"].isin(ALLOWED_CHROMS)]
    df = df.rename(columns={"CopyNumber": sample_name})
    dfs_cn.append(df)
    print(f"  Loaded CN: {get_display_name(sample_name)}")

merged_cn = dfs_cn[0]
for df in dfs_cn[1:]:
    merged_cn = pd.merge(merged_cn, df, on=["Chromosome","Start"], how="inner")
merged_cn = sort_df_by_genome(merged_cn)
out_cn = os.path.join(OUTPUT_DIR, "merged_CN_matrix.tsv")
merged_cn.to_csv(out_cn, sep="\t", index=False)
print(f"\nSaved: {out_cn}  [{merged_cn.shape[0]} bins x {len(dfs_cn)} samples]")
sample_cols_cn = [c for c in merged_cn.columns if c not in ["Chromosome","Start"]]


### 2.2  Filter Unmappable Bins and Apply ENCODE Blacklist (CN)


In [ ]:
df_cn       = pd.read_csv(os.path.join(OUTPUT_DIR, "merged_CN_matrix.tsv"),
                          sep="\t", dtype={"Chromosome": str})
scols_cn    = [c for c in df_cn.columns if c not in ["Chromosome","Start"]]

# Step 1: remove unmappable bins (-1 sentinel)
mask_cn     = (df_cn[scols_cn] == -1).any(axis=1)
df_cn_clean = df_cn[~mask_cn].copy()
print(f"Bins before -1 filter        : {len(df_cn)}")
print(f"Bins after  -1 filter        : {len(df_cn_clean)}  (removed {mask_cn.sum()})")

# Step 2: apply the same ENCODE blacklist used for the log2 CNR matrix.
# Using identical bin sets for both matrices keeps every downstream analysis
# (heterogeneity scores, arm-level aggregation, etc.) comparing exactly the
# same genomic positions.
if 'BLACKLIST' not in dir():
    BLACKLIST = load_blacklist()
n_before_cn = len(df_cn_clean)
df_cn_clean = filter_blacklist(df_cn_clean, BLACKLIST, window_size=WINDOW_SIZE)
print(f"Bins after  blacklist filter : {len(df_cn_clean)}  (removed {n_before_cn - len(df_cn_clean)})")

df_cn_clean = sort_df_by_genome(df_cn_clean)
df_cn_clean.to_csv(os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"), sep="\t", index=False)
print("Saved: filtered_CN_matrix.tsv  (blacklisted, -1 filtered, integer CN)")


### 2.3  Hierarchical Clustering - Integer Copy Number


In [ ]:
def plot_cn_clustermap(metric, method, out_name):
    df_cn_cl = pd.read_csv(os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"),
                           sep="\t", dtype={"Chromosome": str})
    cn_num   = df_cn_cl.drop(columns=["Chromosome","Start"]).apply(pd.to_numeric, errors="coerce")
    samps    = sort_sample_cols(cn_num.columns.tolist())
    cn_num   = cn_num[samps]

    display_names    = [get_display_name(s) for s in samps]
    cn_num_disp      = cn_num.copy()
    cn_num_disp.columns = display_names
    row_colors_cn    = pd.Series([get_color(s) for s in samps], index=display_names)
    legend_cn        = build_legend_patches(samps)
    cn_max_display   = min(int(cn_num.quantile(0.99).max()) + 1, 8)

    g_cn = sns.clustermap(
        cn_num_disp.T, cmap="RdBu_r", metric=metric, method=method,
        vmin=0, vmax=cn_max_display,
        row_cluster=True, col_cluster=False,
        row_colors=row_colors_cn,
        figsize=(16, max(8, len(samps) * 0.55)),
        cbar_pos=None, dendrogram_ratio=(0.12, 0), colors_ratio=0.025,
    )
    g_cn.fig.subplots_adjust(right=0.80)
    g_cn.ax_col_dendrogram.set_visible(False)

    # Row-colour annotation strip draws its own x-axis by default, which shows
    # up as a stray tick beneath the group colour column. Hide it entirely.
    g_cn.ax_row_colors.set_xticks([])
    g_cn.ax_row_colors.tick_params(bottom=False, labelbottom=False)

    bin_info = df_cn_cl[["Chromosome","Start"]].reset_index(drop=True)
    bin_info["idx"] = range(len(bin_info))
    clpos, cltxt, cdivs, prev_c = [], [], [], None
    for chrom, grp in bin_info.groupby("Chromosome", sort=False):
        iv = grp["idx"].values
        clpos.append(float(iv[0]+iv[-1])/2); cltxt.append(str(chrom).replace("chr",""))
        if prev_c is not None: cdivs.append(float(iv[0]))
        prev_c = chrom
    g_cn.ax_heatmap.set_xticks(clpos)
    g_cn.ax_heatmap.set_xticklabels(cltxt, rotation=90, fontsize=max(6, FONT_SIZE - 2))
    g_cn.ax_heatmap.set_xlabel("Chromosome", fontsize=FONT_SIZE)
    for xd in cdivs:
        g_cn.ax_heatmap.axvline(xd, color="black", linestyle="--", alpha=0.25, linewidth=0.5)

    ordered_cn = cn_num_disp.T.index[g_cn.dendrogram_row.reordered_ind]
    g_cn.ax_heatmap.set_yticks(np.arange(len(ordered_cn)) + 0.5)
    g_cn.ax_heatmap.set_yticklabels(ordered_cn, fontsize=FONT_SIZE - 2,
                                rotation=0, va="center")

    for yd in range(1, len(ordered_cn)):
        g_cn.ax_heatmap.axhline(yd, color="black", linestyle="--", alpha=0.25, linewidth=0.5)

    cbar_ax_cn = g_cn.fig.add_axes([0.85, 0.62, 0.018, 0.22])
    norm_cn    = mpl.colors.Normalize(vmin=0, vmax=cn_max_display)
    cb_cn      = mpl.colorbar.ColorbarBase(cbar_ax_cn, cmap=plt.get_cmap("RdBu_r"),
                                           norm=norm_cn, orientation="vertical")
    cb_cn.set_label("Copy Number", fontsize=FONT_SIZE)
    g_cn.fig.legend(handles=legend_cn, title="Group",
                    fontsize=FONT_SIZE - 2, title_fontsize=FONT_SIZE,
                    loc="upper right", bbox_to_anchor=(0.995, 0.52),
                    bbox_transform=g_cn.fig.transFigure, frameon=True, framealpha=0.95)
    g_cn.fig.suptitle(
        f"Hierarchical Clustering - Integer Copy Number ({method.title()} / {metric.title()})",
        y=1.01, fontsize=FONT_SIZE - 2)
    save_fig(g_cn.fig, out_name)
    plt.show()

plot_cn_clustermap("euclidean", "ward",      "clustering_CN_ward_euclidean")
plot_cn_clustermap("correlation", "average", "clustering_CN_avg_correlation")

### 2.4  Individual Sample Genome Plots - Integer CN


In [ ]:
# Reuses plot_individual_samples() defined in Part 1.4
plot_individual_samples(
    os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"),
    value_label="Copy Number", y_range=None,
    output_prefix="CN", title_suffix="Integer Copy Number",
)


### 2.5  Genome-wide CN Distribution per Sample
Modal peak validates the FREEC ploidy assignment.
Fusion clones should show a mode near CN = 4; Control and Parental near CN = 2.


In [ ]:
df_cn_dist = pd.read_csv(os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"),
                         sep="\t", dtype={"Chromosome": str})
scols_dist = sort_sample_cols(
    [c for c in df_cn_dist.columns if c not in ["Chromosome","Start"]]
)
n_samps = len(scols_dist)
ncols_d = 3; nrows_d = (n_samps + ncols_d - 1) // ncols_d
fig, axes = plt.subplots(nrows_d, ncols_d, figsize=(5*ncols_d, 4*nrows_d), sharey=False)
axes = np.array(axes).flatten()

for ax, s in zip(axes, scols_dist):
    vals   = df_cn_dist[s].dropna().astype(int)
    max_cn = max(int(vals.max()), 4)
    bins   = range(0, max_cn + 2)
    ax.bar(list(bins[:-1]), [np.sum(vals == v) for v in bins[:-1]],
           color=get_color(s), edgecolor="black", alpha=0.8, width=0.85)
    med  = float(vals.median())
    mode = int(vals.mode()[0])
    ax.axvline(med,  color="red",    linestyle="--", linewidth=1.4, label=f"Median: {med:.1f}")
    ax.axvline(mode, color="purple", linestyle=":",  linewidth=1.4, label=f"Mode: {mode}")
    ax.set_xticks(range(0, max_cn + 2))
    ax.set_xticklabels(range(0, max_cn + 2), rotation=45, ha="right", fontsize=max(6, FONT_SIZE - 8))
    ax.set_xlabel("Copy Number", fontsize=max(6, FONT_SIZE - 6))
    ax.set_ylabel("Number of Bins", fontsize=max(6, FONT_SIZE - 6))
    ax.set_title(get_display_name(s), fontsize=max(6, FONT_SIZE - 5), color=get_color(s), fontweight="bold")
    ax.legend(fontsize=max(6, FONT_SIZE - 8))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))

for j in range(len(scols_dist), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Genome-wide CN Distribution per Sample\n"
    "(modal shift toward CN=4 indicates FREEC captured a whole-genome doubling; "
    "Parental lines shown as a diploid baseline reference)",
    fontsize=max(6, FONT_SIZE - 4), y=1.02,
)
plt.tight_layout()
save_fig(fig, "CN_distribution")
plt.show()

print(f"\n{'Sample':<10} {'Mode CN':>8} {'Median CN':>10} {'Pct at mode':>12}")
for s in scols_dist:
    vals = df_cn_dist[s].dropna().astype(int)
    mode = int(vals.mode()[0])
    pct  = (vals == mode).mean() * 100
    print(f"{get_display_name(s):<10} {mode:>8} {vals.median():>10.1f} {pct:>11.1f}%")


### 2.6  Per-Chromosome Heatmap - Integer CN


In [ ]:
def plot_chromosome_heatmap_cn(matrix_file, chromosome,
                               vmin=0, vmax=8, cmap="RdBu_r",
                               output_prefix="chr_CN"):
    df        = pd.read_csv(matrix_file, sep="\t", dtype={"Chromosome": str})
    chrom_str = str(chromosome).replace("chr","")
    df_chr    = df[df["Chromosome"].apply(lambda c: str(c).replace("chr","")) == chrom_str].copy()
    if df_chr.empty:
        print(f"No data for chromosome {chrom_str}"); return

    df_chr = sort_df_by_genome(df_chr).reset_index(drop=True)
    scols  = sort_sample_cols([c for c in df_chr.columns if c not in ["Chromosome","Start"]])
    n      = len(scols)
    fig, axes = plt.subplots(n, 1, figsize=(14, max(3, 0.55*n+1.2)), squeeze=False)
    fig.subplots_adjust(hspace=0.05, right=0.86)

    for i, s in enumerate(scols):
        ax  = axes[i, 0]
        img = np.array([df_chr[s].to_numpy(dtype=float)])
        ax.imshow(img, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap, interpolation="nearest")
        ax.set_yticks([]); ax.set_xticks([])
        ax.set_ylabel(get_display_name(s), fontsize=max(6, FONT_SIZE - 7), rotation=0,
                      ha="right", va="center", labelpad=4,
                      color=get_color(s), fontweight="bold")
        for sp in ax.spines.values():
            sp.set_linewidth(0.6); sp.set_edgecolor(get_color(s))

    cbar_ax = fig.add_axes([0.88, 0.12, 0.025, 0.76])
    fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax)),
                 cax=cbar_ax, label="Copy Number")
    fig.suptitle(f"Chromosome {chrom_str} - Integer Copy Number",
                 fontsize=max(6, FONT_SIZE - 4), fontweight="bold", y=1.01)
    save_fig(fig, f"{output_prefix}_chr{chrom_str}")
    plt.show()

for chrom in [str(i) for i in range(1, 23)] + ["X"]:
    plot_chromosome_heatmap_cn(
        os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"),
        chromosome=chrom, vmin=0, vmax=8,
        output_prefix="chr_CN",
    )


---
# Part 3 - Intra-Group CN Heterogeneity Score (Integer Copy Number)

**Formula:** fraction of pairwise CN comparisons that disagree per bin, averaged across the genome.
With 8 clones per group, Control and Fusion each give 28 pairwise comparisons, which is statistically reasonable.
Parental (n=2) gives only 1 pair, so its score is computed but should be treated as low-information.


### 3.1  Calculate Heterogeneity Score


In [ ]:
def calculate_heterogeneity_score(cn_file, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df    = pd.read_csv(cn_file, sep="\t", dtype={"Chromosome": str})
    scols = [c for c in df.columns if c not in ["Chromosome","Start"]]

    group_samples = defaultdict(list)
    for s in scols:
        group_samples[get_group(s)].append(s)

    eligible   = {g: s for g, s in group_samples.items() if len(s) >= 2}
    ineligible = {g: s for g, s in group_samples.items() if len(s) <  2}

    if ineligible:
        print("Groups with only 1 sample (skipped):")
        for g, s in ineligible.items():
            print(f"  {GROUP_NAMES.get(g,g)}: {[get_display_name(x) for x in s]}")

    if not eligible:
        print("\nHeterogeneity score requires >= 2 samples per group.")
        return None, None

    het_matrix = df[["Chromosome","Start"]].copy()
    avg_scores = {}

    print("\nComputing heterogeneity scores:")
    for grp, members in eligible.items():
        rep_vals   = df[members].values.astype(float)
        row_scores = []
        for row in rep_vals:
            pairs  = list(combinations(row, 2))
            n_diff = sum(1 for a, b in pairs if a != b)
            row_scores.append(n_diff / len(pairs) if pairs else 0.0)
        het_matrix[grp] = row_scores
        avg_scores[grp] = float(np.mean(row_scores))
        n_pairs = len(list(combinations(members, 2)))
        print(f"  {GROUP_NAMES.get(grp,grp):<22}  avg = {avg_scores[grp]:.4f}  "
              f"({len(members)} samples, {n_pairs} pairs)")

    het_matrix = sort_df_by_genome(het_matrix)
    het_matrix.to_csv(os.path.join(out_dir, "CN_heterogeneity_per_bin.tsv"), sep="\t", index=False)
    avg_df = pd.DataFrame(
        [(g, avg_scores[g], g, GROUP_NAMES.get(g,g)) for g in avg_scores],
        columns=["Sample","Heterogeneity_Score","Group_Key","Group_Name"],
    )
    avg_df.to_csv(os.path.join(out_dir, "CN_heterogeneity_scores.tsv"), sep="\t", index=False)
    return het_matrix, avg_df

HET_DIR = os.path.join(OUTPUT_DIR, "heterogeneity_score")
het_matrix, het_scores = calculate_heterogeneity_score(
    os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"), HET_DIR
)


### 3.2  Plot Heterogeneity Score


In [ ]:
if het_scores is None:
    print("Heterogeneity plots skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(7, 4))
    ax = axes[0]
    sorted_het = het_scores if CIN_INCLUDE_PARENTAL_IN_STATS else het_scores[het_scores["Group_Key"] != "P"]
    sorted_het = sorted_het.copy()
    sorted_het["_ord"] = sorted_het["Group_Key"].apply(
        lambda g: SAMPLE_GROUP_ORDER.index(g) if g in SAMPLE_GROUP_ORDER else 99)
    sorted_het = sorted_het.sort_values("_ord").drop(columns="_ord")

    ax.bar(sorted_het["Group_Name"], sorted_het["Heterogeneity_Score"],
           color=[GROUP_COLORS.get(g,"gray") for g in sorted_het["Group_Key"]],
           edgecolor="black", linewidth=0.8)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Mean Heterogeneity\nScore", fontsize=max(6, FONT_SIZE))
    ax.set_xlabel("Group", fontsize=max(6, FONT_SIZE - 6))
    # ax.set_title("Genome-wide CN Heterogeneity\n(fraction of pairwise-differing bins per group)", fontsize=max(6, FONT_SIZE - 5))
    ax.tick_params(axis="x", rotation=15)
    # ax.legend(handles=build_legend_patches(sorted_het["Group_Key"].tolist()),
            #   fontsize=max(6, FONT_SIZE - 2), loc="upper right")

    ax2 = axes[1]
    het_bins = pd.read_csv(os.path.join(HET_DIR, "CN_heterogeneity_per_bin.tsv"),
                           sep="\t", dtype={"Chromosome": str})
    if not CIN_INCLUDE_PARENTAL_IN_STATS and 'P' in het_bins.columns:
        het_bins = het_bins.drop(columns=['P'])
    het_bins = sort_df_by_genome(het_bins)
    het_grps = sort_sample_cols([c for c in het_bins.columns if c not in ["Chromosome","Start"]])
    bh, cb_h, lp_h, cl_h = get_chrom_layout(het_bins[["Chromosome","Start"]])
    het_bins = het_bins.merge(bh[["Chromosome","Start","Plot_Index"]],
                              on=["Chromosome","Start"], how="left")
    for g in het_grps:
        ax2.scatter(het_bins["Plot_Index"], het_bins[g],
                    color=GROUP_COLORS.get(g,"gray"), s=4, alpha=0.6,
                    label=GROUP_NAMES.get(g,g))
    for x in cb_h:
        ax2.axvline(x, color="black", linestyle="--", alpha=0.2, linewidth=0.5)
    ax2.set_xticks(lp_h); ax2.set_xticklabels(cl_h, rotation=90, fontsize=max(6, FONT_SIZE - 9))
    ax2.set_ylim(0, 1)
    ax2.set_xlabel("Chromosome", fontsize=max(6, FONT_SIZE - 7)); ax2.set_ylabel("Heterogeneity Score", fontsize=max(6, FONT_SIZE))
    ax2.set_title("Per-bin CN Heterogeneity Score (by group)", fontsize=max(6, FONT_SIZE - 5))
    ax2.legend(fontsize=max(6, FONT_SIZE - 2), loc="upper right")
    ax2.grid(axis="y", linestyle="--", alpha=0.3)

    plt.tight_layout()
    save_fig(fig, "heterogeneity_score_plot")
    plt.show()


### 3.3  Label-Permutation Test for the Heterogeneity Score Gap
The pairwise comparisons that make up the heterogeneity score are not independent
samples (each clone appears in multiple pairs), so a plain t-test on the group
bar heights in 3.2 is not statistically valid. This cell instead shuffles the
Control/Fusion group labels across all 16 clones many times, recomputes the
Fusion-minus-Control heterogeneity gap under each shuffle to build a null
distribution, and reports where the real observed gap falls in that null.


In [ ]:
def _cn_equality_score(arr):
    """
    Mean fraction of pairs (per bin, averaged over bins) that disagree, for an
    (n_bins, n_samples) integer CN array. Uses a count-based identity so it does
    not need to enumerate sample-pairs explicitly, which keeps this fast enough
    to call inside a permutation loop.
    """
    n = arr.shape[1]
    total_pairs = n * (n - 1)
    max_val = int(np.nanmax(arr))
    equal_pairs = np.zeros(arr.shape[0])
    for v in range(0, max_val + 2):
        count_v = (arr == v).sum(axis=1)
        equal_pairs += count_v * (count_v - 1)
    return float(np.mean(1 - equal_pairs / total_pairs))


def run_group_permutation_test(matrix_file, group_a, group_b, score_fn, score_kwargs=None,
                               n_permutations=N_PERMUTATIONS, seed=RANDOM_SEED):
    """
    Label-permutation test for score_fn(group_b) - score_fn(group_a).
    Pools the real samples from both groups, then repeatedly reassigns which
    pooled samples are "pseudo group_b" / "pseudo group_a" (keeping the original
    group sizes), recomputing the gap each time to build a null distribution.
    """
    score_kwargs = score_kwargs or {}
    df = pd.read_csv(matrix_file, sep="\t", dtype={"Chromosome": str})
    scols = [c for c in df.columns if c not in ["Chromosome", "Start"]]
    a_cols = [c for c in scols if get_group(c) == group_a]
    b_cols = [c for c in scols if get_group(c) == group_b]
    n_a, n_b = len(a_cols), len(b_cols)
    if n_a < 2 or n_b < 2:
        raise ValueError(f"Need >=2 samples per group; have {n_a} ({group_a}) and {n_b} ({group_b}).")

    pooled_cols = a_cols + b_cols
    arr_pool = df[pooled_cols].to_numpy(dtype=float)

    observed_a = score_fn(arr_pool[:, :n_a], **score_kwargs)
    observed_b = score_fn(arr_pool[:, n_a:], **score_kwargs)
    observed_gap = observed_b - observed_a

    rng = np.random.default_rng(seed)
    idx = np.arange(n_a + n_b)
    null_gaps = np.empty(n_permutations)
    for i in range(n_permutations):
        rng.shuffle(idx)
        b_idx = idx[:n_b]
        a_idx = idx[n_b:]
        score_b = score_fn(arr_pool[:, b_idx], **score_kwargs)
        score_a = score_fn(arr_pool[:, a_idx], **score_kwargs)
        null_gaps[i] = score_b - score_a

    p_one_sided = (np.sum(null_gaps >= observed_gap) + 1) / (n_permutations + 1)
    p_two_sided = (np.sum(np.abs(null_gaps) >= abs(observed_gap)) + 1) / (n_permutations + 1)

    return {
        "observed_a": observed_a, "observed_b": observed_b, "observed_gap": observed_gap,
        "null_gaps": null_gaps, "p_one_sided": p_one_sided, "p_two_sided": p_two_sided,
        "group_a": group_a, "group_b": group_b, "n_a": n_a, "n_b": n_b,
    }


def plot_permutation_null(perm_result, metric_label, out_name):
    null_gaps = perm_result["null_gaps"]
    observed_gap = perm_result["observed_gap"]
    lbl_a = GROUP_NAMES.get(perm_result["group_a"], perm_result["group_a"])
    lbl_b = GROUP_NAMES.get(perm_result["group_b"], perm_result["group_b"])

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(null_gaps, bins=40, color="lightgray", edgecolor="black", linewidth=0.5)
    ax.axvline(observed_gap, color="crimson", linewidth=2,
               label=f"Observed gap = {observed_gap:.4f}")
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel(f"{metric_label} gap ({lbl_b} minus {lbl_a}), permuted group labels")
    ax.set_ylabel("Permutations")
    ax.set_title(
        f"{metric_label} - Label-Permutation Test\n"
        f"{lbl_b} vs {lbl_a}  (n={perm_result['n_b']} vs n={perm_result['n_a']}, "
        f"{len(null_gaps)} permutations)\n"
        f"one-sided p = {perm_result['p_one_sided']:.4f}   "
        f"two-sided p = {perm_result['p_two_sided']:.4f}"
    )
    ax.legend(loc="upper left")
    if SHOW_GRID:
        ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    save_fig(fig, out_name)
    plt.show()


het_perm_result = run_group_permutation_test(
    os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"),
    group_a="C", group_b="F", score_fn=_cn_equality_score,
)
print(f"Observed CN heterogeneity:  Control={het_perm_result['observed_a']:.4f}  "
      f"Fusion={het_perm_result['observed_b']:.4f}  gap={het_perm_result['observed_gap']:.4f}")
print(f"Permutation p-value (one-sided, Fusion > Control): {het_perm_result['p_one_sided']:.4f}")
print(f"Permutation p-value (two-sided):                   {het_perm_result['p_two_sided']:.4f}")

plot_permutation_null(het_perm_result, "CN Heterogeneity Score", "CN_heterogeneity_permutation_test")


---
# Part 4 - Chromosomal Instability (CIN) Metrics

Six CIN metrics computed in Python, implementing the same formulas as the CINmetrics R package (Oza et al. 2023, Bioinformatics).
All metrics are derived from the `_ratio.txt` files and the ploidy assumptions in `GROUP_PLOIDY` / `SAMPLE_PLOIDY_OVERRIDE` (see the Configuration cell).
**Data used:** integer `CopyNumber` and `MedianRatio` columns, both ploidy-corrected Stage 2 outputs. Bins are filtered with the -1 sentinel and the ENCODE blacklist, matching every other analysis in this notebook.

| Metric | Type | Formula |
|---|---|---|
| **FGA** | Numerical | Fraction of bins where \|CN - ploidy\| >= 1 |
| **TAI** | Overall (signed) | Length-weighted mean log2(CN/ploidy) |
| **mTAI** | Overall (absolute) | Length-weighted mean \|log2(CN/ploidy)\| |
| **BPC** | Structural | Count of CN transitions between adjacent bins (same chromosome) |
| **CNA** | Structural | Count of segment transitions where \|Delta MedianRatio\| >= 0.2 |
| **BaseSegs** | Numerical | Total altered base pairs where \|CN - ploidy\| >= 1 |

**Note on TAI vs mTAI:** TAI is signed, so gains and losses can cancel out. mTAI takes the absolute value and is the preferred overall instability metric here, since post-fusion genomes can have asymmetric gain/loss patterns that TAI would underestimate.

**Parental toggle:** the full CIN metrics table always includes Parental. `CIN_INCLUDE_PARENTAL_IN_STATS` (set in the Configuration cell) controls whether Parental is included in the boxplots and pairwise significance tests below.


### 4.1  Load Raw Ratio Data for CIN Metric Computation


In [ ]:
# Load Chromosome, Start, MedianRatio, CopyNumber from each _ratio.txt.
# This is the only section that reads MedianRatio (needed for the CNA score).
print("Loading raw ratio files for CIN metrics...")
if 'BLACKLIST' not in dir():
    BLACKLIST = load_blacklist()

dfs_cin = {}
for f in sorted(ratio_files):
    sample_name = os.path.basename(f).split("_paired")[0]
    df = pd.read_csv(f, sep="\t",
                     usecols=["Chromosome","Start","MedianRatio","CopyNumber"],
                     dtype={"Chromosome": str})
    df = df[df["Chromosome"].isin(ALLOWED_CHROMS)]
    # Remove unmappable bins (-1 sentinel in either column)
    df = df[(df["CopyNumber"] != -1) & (df["MedianRatio"] != -1)].copy()
    # Apply the same ENCODE blacklist used everywhere else in the notebook
    df = filter_blacklist(df, BLACKLIST, window_size=WINDOW_SIZE)
    df = sort_df_by_genome(df).reset_index(drop=True)
    dfs_cin[sample_name] = df
    print(f"  {get_display_name(sample_name):<8}  {len(df)} bins  ploidy={get_ploidy(sample_name)}")

print(f"\nLoaded {len(dfs_cin)} samples.")


### 4.2  Compute CIN Metrics


In [ ]:
def compute_cin_metrics(dfs_cin_dict, window_size=WINDOW_SIZE, cna_thresh=CNA_THRESHOLD):
    """
    Compute 6 CIN metrics from FREEC _ratio.txt data.
    Implements formulas equivalent to the CINmetrics R package (Oza et al. 2023).

    Returns a DataFrame with one row per sample:
    Sample, Group, Ploidy, FGA, TAI, mTAI, BPC, CNA_Score, BaseSegs
    """
    records = []
    for sample, df in dfs_cin_dict.items():
        ploidy = get_ploidy(sample)
        CN   = df["CopyNumber"].values.astype(float)
        MR   = df["MedianRatio"].values.astype(float)
        chrm = df["Chromosome"].values
        N    = len(CN)
        G    = N * window_size   # total callable genome length in bp

        # -- FGA: Fraction of Genome Altered ----------------------------------
        altered_mask = np.abs(CN - ploidy) >= 1
        FGA = altered_mask.sum() * window_size / G

        # -- TAI and mTAI: Total Aberration Index -----------------------------
        CN_safe  = np.where(CN == 0, 1e-3, CN)
        log2_dev = np.log2(CN_safe / ploidy)
        TAI  = float(np.sum(log2_dev)      * window_size / G)   # signed
        mTAI = float(np.sum(np.abs(log2_dev)) * window_size / G)  # absolute

        # -- BaseSegs: Total altered base pairs -------------------------------
        BaseSegs = int(altered_mask.sum() * window_size)

        # -- BPC: Breakpoint Count ---------------------------------------------
        same_chrom = chrm[:-1] == chrm[1:]
        cn_change  = CN[:-1] != CN[1:]
        BPC = int(np.sum(same_chrom & cn_change))

        # -- CNA Score ---------------------------------------------------------
        mr_change = np.abs(MR[:-1] - MR[1:]) >= cna_thresh
        CNA_score = int(np.sum(same_chrom & mr_change))

        records.append({
            "Sample":    sample,
            "Group":     get_group(sample),
            "Ploidy":    ploidy,
            "FGA":       round(FGA, 6),
            "TAI":       round(TAI, 6),
            "mTAI":      round(mTAI, 6),
            "BPC":       BPC,
            "CNA_Score": CNA_score,
            "BaseSegs":  BaseSegs,
        })
        print(f"  {get_display_name(sample):<6}  FGA={FGA:.4f}  mTAI={mTAI:.4f}  BPC={BPC}  CNA={CNA_score}")

    return pd.DataFrame(records)

print("Computing CIN metrics:")
cin_df = compute_cin_metrics(dfs_cin)
cin_path = os.path.join(OUTPUT_DIR, "CIN_metrics.tsv")
cin_df.to_csv(cin_path, sep="\t", index=False)
print(f"\nSaved: {cin_path}  (all samples, including Parental)")
print(cin_df[["Sample","Group","Ploidy","FGA","TAI","mTAI","BPC","CNA_Score","BaseSegs"]].to_string(index=False))

# Apply the Parental toggle for downstream stats and plots only; the table
# above and the saved TSV always contain every sample.
cin_df_stats = cin_df if CIN_INCLUDE_PARENTAL_IN_STATS else cin_df[cin_df["Group"] != "P"].copy()
if not CIN_INCLUDE_PARENTAL_IN_STATS:
    print("\nCIN_INCLUDE_PARENTAL_IN_STATS is False - Parental excluded from plots and pairwise stats below.")


### 4.3  Pairwise Statistics vs Control


In [ ]:
def compute_cin_pairwise_stats(cin_df_in, metrics, control_group="C", alpha=ALPHA):
    records = []
    ctrl_df = cin_df_in[cin_df_in["Group"] == control_group]
    groups_present = [g for g in SAMPLE_GROUP_ORDER
                      if g in cin_df_in["Group"].unique() and g != control_group]

    for metric in metrics:
        ctrl_vals = ctrl_df[metric].values.astype(float)
        p_raw = {}
        for grp in groups_present:
            grp_vals = cin_df_in[cin_df_in["Group"] == grp][metric].values.astype(float)
            if len(ctrl_vals) < 2 or len(grp_vals) < 2:
                continue
            t_stat, p = ttest_ind(grp_vals, ctrl_vals, equal_var=False)
            p_raw[grp] = (t_stat, p)

        valid_groups = list(p_raw.keys())
        if valid_groups:
            raw_ps = [p_raw[g][1] for g in valid_groups]
            _, adj_ps, _, _ = multipletests(raw_ps, method="fdr_bh")
        else:
            adj_ps = []

        for i, grp in enumerate(valid_groups):
            t_stat, p = p_raw[grp]
            p_adj     = adj_ps[i]
            if   p_adj < 0.001: sig = "***"
            elif p_adj < 0.01:  sig = "**"
            elif p_adj < 0.05:  sig = "*"
            # elif p_adj < 0.1:   sig = "+"
            else:               sig = "ns"
            records.append({
                "Metric":  metric,
                "Group1":  GROUP_NAMES.get(grp, grp),
                "Group2":  GROUP_NAMES.get(control_group, control_group),
                "t_stat":  round(t_stat, 4),
                "p_value": round(p, 4),
                "p_adj":   round(p_adj, 4),
                "sig":     sig,
            })

    return pd.DataFrame(records)

ALL_METRICS = ["FGA", "TAI", "mTAI", "BPC", "CNA_Score", "BaseSegs"]
cin_stat_df = compute_cin_pairwise_stats(cin_df_stats, ALL_METRICS)
cin_stat_path = os.path.join(OUTPUT_DIR, "CIN_metrics_pairwise_stats.tsv")
cin_stat_df.to_csv(cin_stat_path, sep="\t", index=False)
print(cin_stat_df.to_string(index=False))
print(f"\nSaved: {cin_stat_path}")
print("+ = trend (p_adj < 0.10). Any comparison involving Parental has n=2 on one side.")


### 4.4  Plot CIN Metrics (multi-metric boxplots)


In [ ]:
def add_significance_brackets(ax, cin_df_in, metric, stat_df, control_group="C", bracket_step_frac=0.08):
    metric_stats = stat_df[stat_df["Metric"] == metric].copy()
    if metric_stats.empty:
        return
    y_vals  = cin_df_in[metric].values.astype(float)
    y_range = np.nanmax(y_vals) - np.nanmin(y_vals)
    step    = y_range * bracket_step_frac if y_range > 0 else 1.0
    y_max   = np.nanmax(y_vals)

    groups_present = [g for g in SAMPLE_GROUP_ORDER if g in cin_df_in["Group"].unique()]
    group_names_ordered = [GROUP_NAMES.get(g, g) for g in groups_present]
    ctrl_name = GROUP_NAMES.get(control_group, control_group)
    if ctrl_name not in group_names_ordered:
        return
    ctrl_x    = group_names_ordered.index(ctrl_name)
    bracket_y = y_max + step

    for _, row in metric_stats.iterrows():
        grp_name = row["Group1"]
        if grp_name not in group_names_ordered:
            continue
        grp_x  = group_names_ordered.index(grp_name)
        x1, x2 = sorted([ctrl_x, grp_x])
        y      = bracket_y
        tip    = step * 0.15

        ax.plot([x1, x1, x2, x2], [y - tip, y, y, y - tip], color="black", linewidth=1.0)
        ax.text((x1 + x2) / 2, y + step * 0.05, row["sig"],
                ha="center", va="bottom",
                fontsize=max(6, plt.rcParams["font.size"] - 3))
        bracket_y += step * 1.8

    ax.set_ylim(top=bracket_y + step)


def styled_boxplot(ax, data, x, y, hue, palette):
    if BOXPLOT_FILL == "outline":
        sns.boxplot(data=data, x=x, y=y, hue=hue,
                    palette={g: "white" for g in palette},
                    dodge=False, linewidth=1.4,
                    flierprops=dict(marker="o", markersize=4,
                                    markerfacecolor="none",
                                    markeredgecolor="black"),
                    boxprops=dict(edgecolor="black"),
                    whiskerprops=dict(color="black"),
                    capprops=dict(color="black"),
                    medianprops=dict(color="black", linewidth=1.5),
                    ax=ax, legend=False)
    else:
        sns.boxplot(data=data, x=x, y=y, hue=hue,
                    palette=palette,
                    dodge=False, linewidth=1.4,
                    flierprops=dict(marker="o", markersize=4),
                    boxprops=dict(edgecolor="black"),
                    whiskerprops=dict(color="black"),
                    capprops=dict(color="black"),
                    medianprops=dict(color="black", linewidth=1.5),
                    ax=ax, legend=False)
        for patch in ax.patches:
            r, g, b, _ = patch.get_facecolor()
            patch.set_facecolor((r, g, b, BOX_ALPHA))


METRICS_TO_PLOT = ["FGA", "mTAI", "BPC", "CNA_Score", "BaseSegs"]
METRIC_LABELS   = {
    "FGA":       "Fraction of Genome Altered",
    "mTAI":      "Modified TAI (|log2 CN/ploidy|)",
    "BPC":       "Breakpoint Count",
    "CNA_Score": "CNA Score (segment transitions)",
    "BaseSegs":  "Altered Base Pairs",
}

cin_long = cin_df_stats.melt(
    id_vars=["Sample", "Group", "Ploidy"],
    value_vars=ALL_METRICS,
    var_name="Metric", value_name="Value"
)
cin_long["Group_Name"] = cin_long["Group"].map(GROUP_NAMES)

n_metrics = len(METRICS_TO_PLOT)
fig, axes = plt.subplots(1, n_metrics, figsize=(3.5 * n_metrics, 5.5))

for ax, metric in zip(axes, METRICS_TO_PLOT):
    sub = cin_long[cin_long["Metric"] == metric].copy()
    sub["_ord"] = sub["Group"].apply(
        lambda g: SAMPLE_GROUP_ORDER.index(g) if g in SAMPLE_GROUP_ORDER else 99)
    sub     = sub.sort_values("_ord")
    palette = {g: GROUP_COLORS.get(g, "gray") for g in sub["Group"].unique()}

    styled_boxplot(ax, sub, x="Group_Name", y="Value", hue="Group", palette=palette)
    sns.stripplot(data=sub, x="Group_Name", y="Value", hue="Group",
                  palette=palette, dodge=False, jitter=False,
                  size=7, edgecolor="black", linewidth=0.8, ax=ax, legend=False)

    if SHOW_STAT_BRACKETS:
        add_significance_brackets(ax, cin_df_stats, metric, cin_stat_df)

    ax.set_title(METRIC_LABELS.get(metric, metric),
                 fontweight="bold" if FONT_BOLD else "normal")
    ax.set_xlabel("Population")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=30)
    if SHOW_GRID:
        ax.grid(axis="y", linestyle="--", alpha=0.35)
    else:
        ax.grid(False)

parental_note = "Parental included" if CIN_INCLUDE_PARENTAL_IN_STATS else "Parental excluded"
fig.suptitle(
    f"CIN Metrics - Welch t-test vs Control  |  FDR-adjusted (BH)  |  {parental_note}\n"
    "+ p<0.10  * p<0.05  ** p<0.01  *** p<0.001",
    y=1.05
)
plt.tight_layout()
save_fig(fig, "CIN_metrics_boxplot_stats")
plt.show()


### 4.5  Doubling-Null Simulation for FGA and BaseSegs
FGA and BaseSegs both use a fixed absolute threshold (|CN - ploidy| >= 1). That
threshold is not ploidy-invariant: the same proportional measurement noise or
sub-clonal drift produces a smaller absolute CN swing in a diploid genome than
in a tetraploid one (CN 2 to 3 is a 50 percent change; CN 4 to 5 is only 25
percent). So some of the apparent FGA/BaseSegs increase in Fusion could be a
pure rounding artifact of comparing across different baseline ploidies, with no
new biology involved.

This cell builds a null that isolates exactly that artifact. For each real
Control clone:
1. Reconstruct FREEC's own continuous (pre-rounding) copy-number estimate as
   `MedianRatio * ploidy` (the standard Control-FREEC relationship
   `CopyNumber = round(Ratio * ploidy)`, just kept in continuous form).
2. Double that continuous estimate, simulating a perfect whole-genome
   duplication of that exact same measured signal (same noise, same sub-clonal
   drift, no new biology added).
3. Re-round to integer CN against 2x the original ploidy, and compute FGA and
   BaseSegs on the result exactly as `compute_cin_metrics()` does for real
   samples.

If real Fusion FGA/BaseSegs are not distinguishable from this doubling-null,
that is evidence the observed increase is mostly a threshold artifact. If real
Fusion clearly exceeds the doubling-null, that is evidence of genuine excess
instability beyond what doubling alone would produce.


In [ ]:
def simulate_doubling_null(dfs_cin_dict, source_group="C", window_size=WINDOW_SIZE):
    records = []
    for sample, df in dfs_cin_dict.items():
        if get_group(sample) != source_group:
            continue
        ploidy_orig = get_ploidy(sample)
        ploidy_doubled = ploidy_orig * 2

        continuous_cn = df["MedianRatio"].values.astype(float) * ploidy_orig
        continuous_cn_doubled = continuous_cn * 2
        cn_doubled = np.round(continuous_cn_doubled)

        chrm = df["Chromosome"].values
        N = len(cn_doubled)
        G = N * window_size

        altered_mask = np.abs(cn_doubled - ploidy_doubled) >= 1
        FGA = altered_mask.sum() * window_size / G
        BaseSegs = int(altered_mask.sum() * window_size)

        cn_safe = np.where(cn_doubled == 0, 1e-3, cn_doubled)
        log2_dev = np.log2(cn_safe / ploidy_doubled)
        mTAI = float(np.sum(np.abs(log2_dev)) * window_size / G)

        same_chrom = chrm[:-1] == chrm[1:]
        cn_change = cn_doubled[:-1] != cn_doubled[1:]
        BPC = int(np.sum(same_chrom & cn_change))

        records.append({
            "Sample": sample, "Source_Group": source_group,
            "Simulated_Ploidy": ploidy_doubled,
            "FGA": round(FGA, 6), "mTAI": round(mTAI, 6),
            "BPC": BPC, "BaseSegs": BaseSegs,
        })
        print(f"  {get_display_name(sample):<6}  (doubled {ploidy_orig} -> {ploidy_doubled})  "
              f"FGA={FGA:.4f}  BaseSegs={BaseSegs}")

    return pd.DataFrame(records)

print("Simulating whole-genome doubling of each real Control clone:")
doubling_null_df = simulate_doubling_null(dfs_cin, source_group="C")
doubling_null_path = os.path.join(OUTPUT_DIR, "CIN_doubling_null.tsv")
doubling_null_df.to_csv(doubling_null_path, sep="\t", index=False)
print(f"\nSaved: {doubling_null_path}")

fusion_fga = cin_df[cin_df["Group"] == "F"]["FGA"].values.astype(float)
null_fga   = doubling_null_df["FGA"].values.astype(float)
t_fga, p_fga = ttest_ind(fusion_fga, null_fga, equal_var=False)

fusion_baseseg = cin_df[cin_df["Group"] == "F"]["BaseSegs"].values.astype(float)
null_baseseg   = doubling_null_df["BaseSegs"].values.astype(float)
t_bs, p_bs = ttest_ind(fusion_baseseg, null_baseseg, equal_var=False)

print(f"\nReal Fusion vs doubling-null FGA:       t={t_fga:.3f}  p={p_fga:.4f}")
print(f"Real Fusion vs doubling-null BaseSegs:  t={t_bs:.3f}  p={p_bs:.4f}")

null_color = "lightgray"
category_order = ["Real Control", "Doubling-null\n(Control x2)", "Real Fusion"]
category_palette = {
    "Real Control": GROUP_COLORS["C"],
    "Doubling-null\n(Control x2)": null_color,
    "Real Fusion": GROUP_COLORS["F"],
}

plot_df_fga = pd.concat([
    cin_df[cin_df["Group"] == "C"][["Sample", "FGA"]].assign(Category="Real Control"),
    doubling_null_df[["Sample", "FGA"]].assign(Category="Doubling-null\n(Control x2)"),
    cin_df[cin_df["Group"] == "F"][["Sample", "FGA"]].assign(Category="Real Fusion"),
])
plot_df_baseseg = pd.concat([
    cin_df[cin_df["Group"] == "C"][["Sample", "BaseSegs"]].assign(Category="Real Control"),
    doubling_null_df[["Sample", "BaseSegs"]].assign(Category="Doubling-null\n(Control x2)"),
    cin_df[cin_df["Group"] == "F"][["Sample", "BaseSegs"]].assign(Category="Real Fusion"),
])

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, plot_df, metric, p_val in zip(axes, [plot_df_fga, plot_df_baseseg],
                                       ["FGA", "BaseSegs"], [p_fga, p_bs]):
    plot_df = plot_df.copy()
    plot_df["Category"] = pd.Categorical(plot_df["Category"], categories=category_order, ordered=True)
    sns.boxplot(data=plot_df, x="Category", y=metric, hue="Category",
                palette=category_palette, dodge=False, linewidth=1.4,
                flierprops=dict(marker="o", markersize=4),
                boxprops=dict(edgecolor="black"), whiskerprops=dict(color="black"),
                capprops=dict(color="black"), medianprops=dict(color="black", linewidth=1.5),
                ax=ax, legend=False)
    for patch in ax.patches:
        r, g, b, _ = patch.get_facecolor()
        patch.set_facecolor((r, g, b, BOX_ALPHA))
    sns.stripplot(data=plot_df, x="Category", y=metric, hue="Category",
                  palette=category_palette, dodge=False, jitter=False,
                  size=7, edgecolor="black", linewidth=0.8, ax=ax, legend=False)
    ax.set_title(f"{metric}\nFusion vs doubling-null p={p_val:.4f}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
    if SHOW_GRID:
        ax.grid(axis="y", linestyle="--", alpha=0.3)

fig.suptitle(
    "Doubling-Null Simulation\n"
    "Is Fusion FGA/BaseSegs beyond what doubling Control's own noise would produce?",
    y=1.05
)
plt.tight_layout()
save_fig(fig, "CIN_doubling_null_comparison")
plt.show()


---
# Part 5 - Chromosome-Arm Level Analysis

This section aggregates bin-level data to chromosome-arm resolution, which gives cleaner and more
biologically interpretable comparisons than bin-level tests, since averaging hundreds of bins per arm
substantially reduces noise.

**Baseline choice:** Control (C1-C8 mean, n=8) is used as the primary statistical baseline throughout this
section, since it is far more robust than the Parental n=2. Parental is still shown as a supplementary
reference track in the delta heatmap (5.2) and included as its own comparison in the arm-level t-tests (5.4),
but is not used as the reference for the perfect-doubling comparison (5.3), since that comparison is only
meaningful for Fusion clones relative to their un-doubled Control counterparts.

**Five subsections:**
- **5.1** Arm-level aggregation functions (shared infrastructure for 5.2-5.5)
- **5.2** Delta log2 CNR heatmap - Fusion and Parental relative to the Control mean, arm level
- **5.3** Delta CN from perfect doubling heatmap - Fusion relative to 2 x Control mean, arm level
- **5.4** Arm-level Welch t-tests - Fusion vs Control, Fusion vs Parental, Control vs Parental
- **5.5** Arm-level t-test visualisation


### 5.1  Arm-Level Aggregation Functions
Shared infrastructure used by all subsequent cells in this section.
Computes length-weighted mean log2 CNR and mean integer CN per chromosome arm per sample.
Centromere positions are loaded from UCSC (or the hardcoded hg38 fallback defined in Section 2).


In [ ]:
if 'CENTROMERES' not in dir():
    CENTROMERES = load_centromeres(GENOME)

def aggregate_to_arms(matrix_file, centromeres, window_size=WINDOW_SIZE):
    """
    Aggregate per-bin values to chromosome-arm level using length-weighted means.
    Returns (arm_df, arm_order) where arm_df has arms as index, samples as columns.
    """
    df = pd.read_csv(matrix_file, sep="\t", dtype={"Chromosome": str})
    df["End"] = df["Start"] + window_size
    scols = [c for c in df.columns if c not in ["Chromosome","Start","End"]]

    numeric_vals = df[scols].apply(pd.to_numeric, errors="coerce")
    bad = (numeric_vals == -1).any(axis=1) | numeric_vals.isnull().any(axis=1)
    df = df[~bad].copy()

    records = {}
    for chrom, cdf in df.groupby("Chromosome"):
        c = str(chrom).replace("chr","")
        if c not in centromeres:
            continue
        cen = centromeres[c]
        for arm_label, mask in [("p", cdf["End"]   <= cen),
                                 ("q", cdf["Start"] >= cen)]:
            arm_df_chunk = cdf[mask]
            if arm_df_chunk.empty:
                continue
            lengths = (arm_df_chunk["End"] - arm_df_chunk["Start"]).values.astype(float)
            arm_key = f"{c}{arm_label}"
            arm_row = {}
            for s in scols:
                vals = arm_df_chunk[s].values.astype(float)
                finite = np.isfinite(vals)
                if finite.sum() == 0:
                    arm_row[s] = np.nan
                else:
                    arm_row[s] = np.average(vals[finite], weights=lengths[finite])
            records[arm_key] = arm_row

    arm_df = pd.DataFrame(records).T
    arm_df.index.name = "Arm"

    def arm_sort_key(arm):
        num = ''.join(filter(str.isdigit, arm))
        letter = arm[-1]
        return (int(num) if num.isdigit() else 99, 0 if letter == "p" else 1)

    arm_order = sorted(arm_df.index.tolist(), key=arm_sort_key)
    arm_df = arm_df.loc[arm_order]
    return arm_df, arm_order

def group_mean_arm(arm_matrix, group_key):
    """Mean across all samples belonging to group_key, per arm. Returns (Series, cols_used)."""
    cols = [c for c in arm_matrix.columns if get_group(c) == group_key]
    if not cols:
        raise ValueError(f"No samples found for group '{group_key}' in arm matrix.")
    return arm_matrix[cols].mean(axis=1), cols

print("Aggregating log2 CNR to arm level...")
arm_log2cnr, arm_order = aggregate_to_arms(
    os.path.join(OUTPUT_DIR, "log2_filtered_ratio_matrix.tsv"), CENTROMERES, window_size=WINDOW_SIZE
)

print("Aggregating integer CN to arm level...")
arm_cn, _ = aggregate_to_arms(
    os.path.join(OUTPUT_DIR, "filtered_CN_matrix.tsv"), CENTROMERES, window_size=WINDOW_SIZE
)

arm_log2cnr.to_csv(os.path.join(OUTPUT_DIR, "arm_log2cnr_matrix.tsv"), sep="\t")
arm_cn.to_csv(os.path.join(OUTPUT_DIR, "arm_CN_matrix.tsv"), sep="\t")
print(f"\nArm-level matrices saved. {len(arm_order)} arms x {arm_log2cnr.shape[1]} samples.")


### 5.2  Delta Log2 CNR Heatmap (Arm Level)
For each Fusion clone and each Parental sample, compute the arm-level delta relative to the Control mean:

Delta log2 CNR (arm) = log2 CNR (arm, sample) - log2 CNR (arm, Control mean)

**Interpretation:** positive (red) means that arm carries proportionally more DNA than the Control baseline;
negative (blue) means less. Parental rows are shown for visual reference only and are not statistically tested here (see 5.4 for the tested Control vs Parental comparison).


In [ ]:
plt.close("all")

control_mean_log2, control_cols_log2 = group_mean_arm(arm_log2cnr, "C")
print(f"Control baseline (n={len(control_cols_log2)}): {[get_display_name(c) for c in control_cols_log2]}")

non_control_cols = sort_sample_cols([c for c in arm_log2cnr.columns if get_group(c) == "F"])
delta_log2 = pd.DataFrame(
    {c: arm_log2cnr[c] - control_mean_log2 for c in non_control_cols},
    index=arm_order
).T
delta_log2.index = [get_display_name(c) for c in delta_log2.index]
delta_log2.to_csv(os.path.join(OUTPUT_DIR, "delta_log2cnr_arm_matrix.tsv"), sep="\t")

raw_max = float(delta_log2.abs().max().max())
vabs    = max(0.5, np.ceil(raw_max * 2) / 2)
print(f"Delta log2CNR range: {delta_log2.min().min():.3f} to {delta_log2.max().max():.3f}")
print(f"Colour scale set to +/-{vabs}")

n_arms = len(arm_order)
n_rows = len(non_control_cols)
fig, ax = plt.subplots(figsize=(max(16, n_arms * 0.38), max(4, n_rows * 0.35 + 1.5)))

im = ax.imshow(delta_log2.values, aspect="auto",
               cmap="RdBu_r", vmin=-vabs, vmax=vabs, interpolation="nearest")

ax.set_xticks(range(n_arms))
ax.set_xticklabels(arm_order, rotation=90, fontsize=max(6, plt.rcParams["xtick.labelsize"] - 1))
ax.set_yticks(range(n_rows))
ax.set_yticklabels(delta_log2.index, fontsize=plt.rcParams["ytick.labelsize"])
row_colors_arm = [get_color(c) for c in non_control_cols]
for tick, col in zip(ax.get_yticklabels(), row_colors_arm):
    tick.set_color(col)
ax.set_xlabel("Chromosome Arm")
ax.set_title("Delta log2 CNR (sample minus Control mean) - Arm level\n"
             "Positive (red) = over-duplication relative to Control baseline  |  "
             "Negative (blue) = under-duplication")

prev_chrom = None
for i, arm in enumerate(arm_order):
    c = ''.join(filter(str.isdigit, arm))
    if c != prev_chrom:
        if prev_chrom is not None:
            ax.axvline(i - 0.5, color="black", linewidth=0.8, alpha=0.5)
        prev_chrom = c
    elif arm.endswith("p"):
        ax.axvline(i + 0.5, color="gray", linewidth=0.4, linestyle="--", alpha=0.4)

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
chrom_tick_pos, chrom_tick_lbls = [], []
prev_c2, start_i2 = None, 0
for i, arm in enumerate(arm_order):
    c = ''.join(filter(str.isdigit, arm))
    if c != prev_c2:
        if prev_c2 is not None:
            chrom_tick_pos.append((start_i2 + i - 1) / 2)
            chrom_tick_lbls.append(prev_c2)
        start_i2, prev_c2 = i, c
if prev_c2:
    chrom_tick_pos.append((start_i2 + len(arm_order) - 1) / 2)
    chrom_tick_lbls.append(prev_c2)
ax2.set_xticks(chrom_tick_pos)
ax2.set_xticklabels(chrom_tick_lbls, fontsize=max(6, plt.rcParams["xtick.labelsize"] - 1))
ax2.set_xlabel("Chromosome")

plt.tight_layout()
fig.canvas.draw()
ax_pos  = ax.get_position()
cbar_ax = fig.add_axes([ax_pos.x1 + 0.008, ax_pos.y0, 0.012, ax_pos.height])
norm    = mpl.colors.Normalize(vmin=-vabs, vmax=vabs)
cb      = mpl.colorbar.ColorbarBase(cbar_ax, cmap=plt.get_cmap("RdBu_r"),
                                     norm=norm, orientation="vertical")
cb.set_label(r"Mean $\Delta$log$_2$ CNR", fontsize=plt.rcParams["legend.fontsize"])
tick_step = 0.25 if vabs <= 0.5 else 0.5 if vabs <= 2 else 1.0
cb.set_ticks(np.arange(-vabs, vabs + tick_step * 0.01, tick_step))
cb.ax.tick_params(labelsize=plt.rcParams["ytick.labelsize"])

save_fig(fig, "delta_log2cnr_arm_heatmap")
plt.show()
plt.close(fig)

### 5.3  Delta CN from Perfect Doubling Heatmap (Arm Level)
Constructs a perfectly doubled null genome by multiplying the Control arm-level mean CN by 2, then
subtracts this from each Fusion clone's arm-level mean CN:

Delta CN (arm) = CN (arm, Fusion) - 2 x CN (arm, Control mean)

**Interpretation:** a value of 0 means that arm doubled exactly as expected, with no secondary CN evolution
above or below the fusion event. Positive values indicate additional copies gained beyond a simple doubling;
negative values indicate copies lost relative to a simple doubling. This comparison is scoped to Fusion clones
only - it is not meaningful for Parental, which was never expected to double.


In [ ]:
plt.close("all")

control_mean_cn, control_cols_cn = group_mean_arm(arm_cn, "C")
perfect_double = control_mean_cn * 2

fusion_cols_cn = sort_sample_cols([c for c in arm_cn.columns if get_group(c) == "F"])
delta_cn = pd.DataFrame(
    {c: arm_cn[c] - perfect_double for c in fusion_cols_cn},
    index=arm_order
).T
delta_cn.index = [get_display_name(c) for c in delta_cn.index]
delta_cn.to_csv(os.path.join(OUTPUT_DIR, "delta_cn_from_perfect_doubling_arm.tsv"), sep="\t")

raw_max_cn = float(delta_cn.abs().max().max())
vabs_cn    = max(0.5, np.ceil(raw_max_cn * 2) / 2)
print(f"Delta CN range: {delta_cn.min().min():.3f} to {delta_cn.max().max():.3f}")
print(f"Colour scale set to +/-{vabs_cn}")

n_arms_cn = len(arm_order)
n_pops_cn = len(fusion_cols_cn)
fig, ax   = plt.subplots(figsize=(max(16, n_arms_cn * 0.38), max(3, n_pops_cn * 0.35 + 1.5)))

im = ax.imshow(delta_cn.values, aspect="auto",
               cmap="RdBu_r", vmin=-vabs_cn, vmax=vabs_cn, interpolation="nearest")

ax.set_xticks(range(n_arms_cn))
ax.set_xticklabels(arm_order, rotation=90, fontsize=max(6, plt.rcParams["xtick.labelsize"] - 1))
ax.set_yticks(range(n_pops_cn))
ax.set_yticklabels(delta_cn.index, fontsize=plt.rcParams["ytick.labelsize"])
ax.set_xlabel("Chromosome Arm")
ax.set_title("Delta CN from Perfect Doubling (Fusion minus 2 x Control mean) - Arm level\n"
             "Positive (red) = gained above expected doubling  |  "
             "Negative (blue) = lost relative to expected doubling")

prev_chrom = None
for i, arm in enumerate(arm_order):
    c = ''.join(filter(str.isdigit, arm))
    if c != prev_chrom:
        if prev_chrom is not None:
            ax.axvline(i - 0.5, color="black", linewidth=0.8, alpha=0.5)
        prev_chrom = c
    elif arm.endswith("p"):
        ax.axvline(i + 0.5, color="gray", linewidth=0.4, linestyle="--", alpha=0.4)

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
chrom_tick_pos2, chrom_tick_lbls2 = [], []
prev_c2, start_i2 = None, 0
for i, arm in enumerate(arm_order):
    c = ''.join(filter(str.isdigit, arm))
    if c != prev_c2:
        if prev_c2 is not None:
            chrom_tick_pos2.append((start_i2 + i - 1) / 2)
            chrom_tick_lbls2.append(prev_c2)
        start_i2, prev_c2 = i, c
if prev_c2:
    chrom_tick_pos2.append((start_i2 + len(arm_order) - 1) / 2)
    chrom_tick_lbls2.append(prev_c2)
ax2.set_xticks(chrom_tick_pos2)
ax2.set_xticklabels(chrom_tick_lbls2, fontsize=max(6, plt.rcParams["xtick.labelsize"] - 1))
ax2.set_xlabel("Chromosome")

plt.tight_layout()
fig.canvas.draw()
ax_pos  = ax.get_position()
cbar_ax = fig.add_axes([ax_pos.x1 + 0.008, ax_pos.y0, 0.012, ax_pos.height])
norm    = mpl.colors.Normalize(vmin=-vabs_cn, vmax=vabs_cn)
cb      = mpl.colorbar.ColorbarBase(cbar_ax, cmap=plt.get_cmap("RdBu_r"),
                                     norm=norm, orientation="vertical")
cb.set_label("Mean Delta CN per arm", fontsize=plt.rcParams["legend.fontsize"])
tick_step = 0.5 if vabs_cn <= 2 else 1.0
cb.set_ticks(np.arange(-vabs_cn, vabs_cn + tick_step * 0.01, tick_step))
cb.ax.tick_params(labelsize=plt.rcParams["ytick.labelsize"])

save_fig(fig, "delta_cn_perfect_doubling_arm_heatmap")
plt.show()
plt.close(fig)


### 5.4  Arm-Level Welch T-Tests
Operates on arm-level mean log2 CNR values rather than individual bins.
Runs the same three comparisons: Fusion vs Control, Fusion vs Parental, Control vs Parental.
Averaging hundreds of bins per arm substantially reduces noise compared to bin-level tests.
FDR correction (BH) is applied across all arms within each comparison.


In [ ]:
ARM_TTEST_DIR = os.path.join(OUTPUT_DIR, "arm_ttest_outputs")
os.makedirs(ARM_TTEST_DIR, exist_ok=True)

def run_arm_ttest(arm_matrix, g1_cols, g2_cols, lbl1, lbl2, out_dir, alpha=ALPHA):
    if len(g1_cols) < 2 or len(g2_cols) < 2:
        print(f"  SKIPPED {lbl1} vs {lbl2}  (need >=2 each; have {len(g1_cols)} vs {len(g2_cols)})")
        return None
    print(f"  {lbl1}_vs_{lbl2}  (n={len(g1_cols)} vs n={len(g2_cols)})")
    records = []
    for arm in arm_matrix.index:
        g1 = arm_matrix.loc[arm, g1_cols].values.astype(float)
        g2 = arm_matrix.loc[arm, g2_cols].values.astype(float)
        g1 = g1[np.isfinite(g1)]; g2 = g2[np.isfinite(g2)]
        if len(g1) < 2 or len(g2) < 2:
            records.append({"Arm": arm, f"{lbl1}_Mean": np.nan, f"{lbl2}_Mean": np.nan,
                             "Delta_log2CNR": np.nan, "t_stat": np.nan, "p_value": np.nan})
            continue
        t_stat, p = ttest_ind(g1, g2, equal_var=False)
        records.append({"Arm": arm, f"{lbl1}_Mean": g1.mean(), f"{lbl2}_Mean": g2.mean(),
                         "Delta_log2CNR": g1.mean() - g2.mean(),
                         "t_stat": round(t_stat, 4), "p_value": round(p, 4)})

    df = pd.DataFrame(records)
    valid = df["p_value"].notna()
    if valid.sum() > 0:
        _, adj, _, _ = multipletests(df.loc[valid, "p_value"], method="fdr_bh")
        df.loc[valid, "p_adj"] = adj
        df.loc[valid, "sig"] = df.loc[valid, "p_adj"].apply(
            lambda p: "***" if p < 0.001 else "**" if p < 0.01
                       else "*" if p < 0.05 else "+" if p < 0.1 else "ns")
    df = df.sort_values("p_adj")
    comp_name = f"{lbl1}_vs_{lbl2}"
    df.to_csv(os.path.join(out_dir, f"{comp_name}_arm_ttest.tsv"), sep="\t", index=False)
    sig_arms = df[df["sig"].isin(["*","**","***","+"])] if "sig" in df.columns else pd.DataFrame()
    print(f"    {len(sig_arms)} arms with p_adj < 0.10")
    return df

print("Running arm-level Welch t-tests (log2 CNR):")
fusion_cols_log2   = sort_sample_cols([c for c in arm_log2cnr.columns if get_group(c) == "F"])
control_cols_log2  = sort_sample_cols([c for c in arm_log2cnr.columns if get_group(c) == "C"])
parental_cols_log2 = sort_sample_cols([c for c in arm_log2cnr.columns if get_group(c) == "P"])

arm_ttest_results = {}

r1 = run_arm_ttest(arm_log2cnr, fusion_cols_log2, control_cols_log2, "Fusion", "Control", ARM_TTEST_DIR)
if r1 is not None: arm_ttest_results["Fusion_vs_Control"] = r1

r2 = run_arm_ttest(arm_log2cnr, fusion_cols_log2, parental_cols_log2, "Fusion", "Parental", ARM_TTEST_DIR)
if r2 is not None: arm_ttest_results["Fusion_vs_Parental"] = r2

r3 = run_arm_ttest(arm_log2cnr, control_cols_log2, parental_cols_log2, "Control", "Parental", ARM_TTEST_DIR)
if r3 is not None: arm_ttest_results["Control_vs_Parental"] = r3

print(f"\nResults saved to: {ARM_TTEST_DIR}")


### 5.5  Arm-Level T-Test Visualisation
Two plot types per comparison:
1. Mean log2 CNR per arm, both groups plotted, significant arms highlighted
2. Delta log2 CNR per arm, directional bar chart, coloured by significance and direction

Chromosome-arm labels replace genomic bin positions on the x-axis.


In [ ]:
plt.close("all")

def plot_arm_ttest_means(ttest_df, sig_threshold=ALPHA, title=""):
    df = ttest_df.copy().reset_index(drop=True)
    mean_cols = [c for c in df.columns if c.endswith("_Mean")]
    lbl1_col, lbl2_col = mean_cols
    lbl1 = lbl1_col.replace("_Mean",""); lbl2 = lbl2_col.replace("_Mean","")

    def arm_sort_key(arm):
        num = ''.join(filter(str.isdigit, str(arm)))
        letter = str(arm)[-1]
        return (int(num) if num.isdigit() else 99, 0 if letter=="p" else 1)
    df["_sk"] = df["Arm"].apply(arm_sort_key)
    df = df.sort_values("_sk").reset_index(drop=True)
    df["x"] = range(len(df))

    sig_mask = df["p_adj"] < sig_threshold
    fig, ax  = plt.subplots(figsize=(max(14, len(df)*0.42), 5))

    ax.scatter(df.loc[~sig_mask,"x"], df.loc[~sig_mask, lbl2_col],
               color="gray", s=28, alpha=0.55, label=f"{lbl2} (ns)")
    ax.scatter(df.loc[sig_mask,"x"],  df.loc[sig_mask,  lbl2_col],
               color="seagreen", s=40, alpha=0.85, label=f"{lbl2} (FDR<{sig_threshold})")
    ax.scatter(df.loc[~sig_mask,"x"], df.loc[~sig_mask, lbl1_col],
               color="lightcoral", s=28, alpha=0.55, label=f"{lbl1} (ns)")
    ax.scatter(df.loc[sig_mask,"x"],  df.loc[sig_mask,  lbl1_col],
               color="crimson", s=40, alpha=0.85, label=f"{lbl1} (FDR<{sig_threshold})")

    y_min = df[[lbl1_col, lbl2_col]].min().min()
    y_max = df[[lbl1_col, lbl2_col]].max().max()
    y_rng = y_max - y_min if y_max != y_min else 1.0

    if SHOW_STAT_BRACKETS and sig_mask.any():
        ax.set_ylim(y_min - y_rng * 0.08, y_max + y_rng * 0.22)
        for _, row in df[sig_mask].iterrows():
            ax.text(row["x"], y_max + y_rng * 0.05, row["sig"],
                    ha="center", va="bottom",
                    fontsize=max(6, plt.rcParams["font.size"] - 3))
    else:
        ax.set_ylim(y_min - y_rng * 0.08, y_max + y_rng * 0.12)

    prev_c = None
    for _, row in df.iterrows():
        c = ''.join(filter(str.isdigit, str(row["Arm"])))
        if c != prev_c:
            if prev_c is not None:
                ax.axvline(row["x"]-0.5, color="black", linestyle="--", alpha=0.3, linewidth=0.6)
            prev_c = c

    ax.set_xticks(df["x"]); ax.set_xticklabels(df["Arm"], rotation=90)
    ax.set_xlabel("Chromosome Arm"); ax.set_ylabel(r"Mean log$_2$ CNR")
    ax.set_title(title)
    ax.legend(loc="upper right",fontsize=(FONT_SIZE - 2))
    if SHOW_GRID: ax.grid(axis="y", linestyle="--", alpha=0.35)
    else:         ax.grid(False)
    plt.tight_layout()
    safe = title.replace(" ","_").replace("/","_")
    save_fig(fig, f"arm_ttest_means_{safe}")
    plt.show(); plt.close(fig)


def plot_arm_ttest_delta(ttest_df, sig_threshold=ALPHA, title=""):
    df = ttest_df.copy().reset_index(drop=True)
    mean_cols = [c for c in df.columns if c.endswith("_Mean")]
    lbl1_col, lbl2_col = mean_cols

    def arm_sort_key(arm):
        num = ''.join(filter(str.isdigit, str(arm)))
        letter = str(arm)[-1]
        return (int(num) if num.isdigit() else 99, 0 if letter=="p" else 1)
    df["_sk"] = df["Arm"].apply(arm_sort_key)
    df = df.sort_values("_sk").reset_index(drop=True)
    df["x"] = range(len(df))

    sig_mask = df["p_adj"] < sig_threshold
    colors   = np.where(sig_mask,
                        np.where(df["Delta_log2CNR"] > 0, "crimson", "steelblue"),
                        "lightgray")

    fig, ax = plt.subplots(figsize=(max(14, len(df)*0.42), 4.5))
    ax.bar(df["x"], df["Delta_log2CNR"], color=colors, width=0.8, linewidth=0)
    ax.axhline(0, color="black", linewidth=0.8)

    y_vals = df["Delta_log2CNR"].values
    y_max  = float(np.nanmax(y_vals)) if np.isfinite(y_vals).any() else 1.0
    y_min  = float(np.nanmin(y_vals)) if np.isfinite(y_vals).any() else -1.0
    y_rng  = y_max - y_min if y_max != y_min else 1.0

    if SHOW_STAT_BRACKETS and sig_mask.any():
        ax.set_ylim(y_min - y_rng * 0.28, y_max + y_rng * 0.28)
        for _, row in df[sig_mask].iterrows():
            y      = row["Delta_log2CNR"]
            offset = y_rng * 0.06
            if y >= 0:
                ax.text(row["x"], y + offset, row["sig"], ha="center", va="bottom",
                        fontsize=max(6, plt.rcParams["font.size"] - 3))
            else:
                ax.text(row["x"], y - offset, row["sig"], ha="center", va="top",
                        fontsize=max(6, plt.rcParams["font.size"] - 3))
    else:
        ax.set_ylim(y_min - y_rng * 0.08, y_max + y_rng * 0.12)

    prev_c = None
    for _, row in df.iterrows():
        c = ''.join(filter(str.isdigit, str(row["Arm"])))
        if c != prev_c:
            if prev_c is not None:
                ax.axvline(row["x"]-0.5, color="black", linestyle="--", alpha=0.3, linewidth=0.6)
            prev_c = c

    ax.set_xticks(df["x"]); ax.set_xticklabels(df["Arm"], rotation=90)
    ax.set_xlabel("Chromosome Arm")
    ax.set_ylabel(r"$\Delta$log$_2$ CNR")
    ax.set_title(f"{title}  |  red=over-duplicated  blue=under-duplicated  gray=ns")
    handles = [Patch(color="crimson",   label="Over-duplicated (sig)"),
               Patch(color="steelblue", label="Under-duplicated (sig)"),
               Patch(color="lightgray", label="Not significant")]
    ax.legend(handles=handles, loc="upper left",fontsize=(FONT_SIZE - 3))
    if SHOW_GRID: ax.grid(axis="y", linestyle="--", alpha=0.3)
    else:         ax.grid(False)
    plt.tight_layout()
    safe = title.replace(" ","_").replace("/","_")
    save_fig(fig, f"arm_ttest_delta_{safe}")
    plt.show(); plt.close(fig)


for comp_name, res_df in arm_ttest_results.items():
    print(f"\n-- {comp_name} --")
    plot_arm_ttest_means(res_df, title=comp_name.replace("_"," "))
    plot_arm_ttest_delta(res_df, title=comp_name.replace("_"," "))
